# Analisis Penyerapan Stokastik (*First Step Analysis*) Rantai Markov
**Mata Kuliah:** Pengantar Proses Stokastik  
**Dosen Pengampu:** Ade Susanti, S.Si., M.Si.  
**Kelompok 1:**
- Achika Vigo Azhyra (M0125001)
- Fadhila Hardi Ningrum (M0124004)
- Fawwaz Absyar Rifai (M0125044)
- Karunia Febyayu Puspitaningtyas (M0124010)
- Ramadhan Imanur Rochim (M0124015)

---

## 1. Landasan Teoretis First Step Analysis (FSA)

*First Step Analysis* (FSA) mengevaluasi fungsional rantai Markov dengan mengkondisikan kejadian pada transisi pertama ($X_1$), lalu menerapkan **Hukum Probabilitas Total** (*Law of Total Probability*) dan **Sifat Markov** (*Markov Property*).

Misalkan $T = \\min\\{n \\ge 0 \\mid X_n \\in S_A\\}$ adalah waktu acak menuju penyerapan (*time to absorption*).

### A. Sistem Persamaan Peluang Penyerapan ($u_i$)
Untuk suatu keadaan penyerap target $a^* \\in S_A$, didefinisikan:
$$u_i = \\Pr\\{X_T = a^* \\mid X_0 = i\\}, \\quad \\forall i \\in S_T$$
Melalui dekomposisi langkah pertama:
$$u_i = P_{ia^*} + \\sum_{k \\in S_T} P_{ik} u_k, \\quad \\forall i \\in S_T$$

### B. Sistem Persamaan Waktu Rata-rata Penyerapan ($v_i$)
Didefinisikan ekspektasi durasi sampai proses terperangkap ke dalam himpunan penyerap:
$$v_i = \\mathbb{E}[T \\mid X_0 = i], \\quad \\forall i \\in S_T$$
Karena langkah pertama menghabiskan tepat 1 satuan waktu:
$$v_i = 1 + \\sum_{k \\in S_T} P_{ik} v_k, \\quad \\forall i \\in S_T$$

Notebook ini bertujuan untuk:
1. Membentuk dan menyelesaikan sistem persamaan $u_i$ dan $v_i$ secara analitis simbolik (SymPy).
2. Memverifikasi kesesuaian solusi dengan Teori Matriks Fundamental Kemeny-Snell $N = (I - Q)^{-1}$.
3. Memvisualisasikan probabilitas penyerapan dan durasi rata-rata dalam bentuk heatmap dan grafik batang.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 120

sys.path.append(os.path.abspath("../scripts"))
import fsa_solver as fsa

print("Pustaka analitis SymPy dan modul FSA Solver berhasil dimuat.")


Pustaka analitis SymPy dan modul FSA Solver berhasil dimuat.


## 2. Pemuatan Model dan Partisi Bentuk Kanonik

Matriks transisi dipartisi ke dalam bentuk kanonik standar:
$$P = \\begin{pmatrix} Q & R \\\\ \\mathbf{0} & I \\end{pmatrix}$$
di mana $Q$ adalah interaksi antarkeadaan transien dan $R$ adalah transisi dari transien menuju penyerap.


In [2]:
model = fsa.get_covid_clinical_model()
trans_names = model["transient_labels"]
absorb_names = model["absorbing_labels"]

print("Submatriks Q (Transien ke Transien):")
display(pd.DataFrame(model["P_float"][:3, :3], index=trans_names, columns=trans_names))

print("\nSubmatriks R (Transien ke Penyerap):")
display(pd.DataFrame(model["P_float"][:3, 3:], index=trans_names, columns=absorb_names))


Submatriks Q (Transien ke Transien):
                Ringan (T1)  Sedang (T2)  Berat/ICU (T3)
Ringan (T1)            0.25         0.05            0.00
Sedang (T2)            0.10         0.30            0.10
Berat/ICU (T3)         0.00         0.20            0.25

Submatriks R (Transien ke Penyerap):
                Sembuh (A1)  Meninggal (A2)
Ringan (T1)            0.69            0.01
Sedang (T2)            0.45            0.05
Berat/ICU (T3)         0.25            0.30


## 3. Penurunan dan Solusi Sistem Persamaan $v_i$ (Rata-rata Waktu Penyerapan)

Sistem persamaan linear waktu rata-rata penyerapan adalah:
$$\\begin{cases}
v_1 = 1 + \\frac{1}{4}v_1 + \\frac{1}{20}v_2 \\\\
v_2 = 1 + \\frac{1}{10}v_1 + \\frac{3}{10}v_2 + \\frac{1}{10}v_3 \\\\
v_3 = 1 + \\frac{1}{5}v_2 + \\frac{1}{4}v_3
\\end{cases}$$


In [3]:
sym_res = fsa.solve_fsa_symbolic_system(model["Q_sym"], model["R_sym"], model["state_labels"])

print("Sistem Persamaan v_i yang Dibangun:")
for eq in sym_res["v_equations"]:
    print("  ", eq)

print("\nSolusi Fraksi Eksak v_i:")
v_sol = sym_res["v_solutions"]
print(v_sol)

v_vars = list(v_sol.keys())
v_weeks = [float(v_sol[k]) for k in v_vars]
v_days = [w * 7 for w in v_weeks]

df_v_result = pd.DataFrame({
    "Keadaan Awal (State)": trans_names,
    "Solusi Fraksi": [str(v_sol[k]) for k in v_vars],
    "Durasi Rata-rata (Minggu)": v_weeks,
    "Durasi Rata-rata (Hari)": v_days
})
display(df_v_result)


Sistem Persamaan v_i yang Dibangun:
   Eq(v_1, v_1/4 + v_2/20 + 1)
   Eq(v_2, v_1/10 + 3*v_2/10 + v_3/10 + 1)
   Eq(v_3, v_2/5 + v_3/4 + 1)

Solusi Fraksi Eksak v_i:
{v_1: 73/50, v_2: 19/10, v_3: 46/25}
  Keadaan Awal (State)  ... Durasi Rata-rata (Hari)
0          Ringan (T1)  ...                   10.22
1          Sedang (T2)  ...                   13.30
2       Berat/ICU (T3)  ...                   12.88

[3 rows x 4 columns]


## 4. Penurunan dan Solusi Sistem Persamaan $u_i$ (Peluang Penyerapan)

### A. Peluang Fatalitas / Kematian ($u_i^{(A_2)}$)
Sistem persamaan peluang terserap ke keadaan Meninggal Dunia ($A_2$):
$$\\begin{cases}
u_1 = \\frac{1}{100} + \\frac{1}{4}u_1 + \\frac{1}{20}u_2 \\\\
u_2 = \\frac{1}{20} + \\frac{1}{10}u_1 + \\frac{3}{10}u_2 + \\frac{1}{10}u_3 \\\\
u_3 = \\frac{3}{10} + \\frac{1}{5}u_2 + \\frac{1}{4}u_3
\\end{cases}$$


In [4]:
print("Sistem Persamaan u_i (Meninggal Dunia):")
for eq in sym_res["u_death_equations"]:
    print("  ", eq)

print("\nSolusi Fraksi Eksak Peluang Meninggal:")
u_d_sol = sym_res["u_death_solutions"]
print(u_d_sol)


Sistem Persamaan u_i (Meninggal Dunia):
   Eq(u_1^{A2}, u_1^{A2}/4 + u_2^{A2}/20 + 1/100)
   Eq(u_2^{A2}, u_1^{A2}/10 + 3*u_2^{A2}/10 + u_3^{A2}/10 + 1/20)
   Eq(u_3^{A2}, u_2^{A2}/5 + u_3^{A2}/4 + 3/10)

Solusi Fraksi Eksak Peluang Meninggal:
{u_1^{A2}: 337/15000, u_2^{A2}: 137/1000, u_3^{A2}: 1637/3750}


### B. Peluang Kesembuhan ($u_i^{(A_1)}$)
Sistem persamaan peluang terserap ke keadaan Sembuh ($A_1$):
$$\\begin{cases}
u_1 = \\frac{69}{100} + \\frac{1}{4}u_1 + \\frac{1}{20}u_2 \\\\
u_2 = \\frac{9}{20} + \\frac{1}{10}u_1 + \\frac{3}{10}u_2 + \\frac{1}{10}u_3 \\\\
u_3 = \\frac{1}{4} + \\frac{1}{5}u_2 + \\frac{1}{4}u_3
\\end{cases}$$


In [5]:
print("Solusi Fraksi Eksak Peluang Sembuh:")
u_r_sol = sym_res["u_recov_solutions"]
print(u_r_sol)

u_d_keys = list(u_d_sol.keys())
u_r_keys = list(u_r_sol.keys())

df_u_summary = pd.DataFrame({
    "Keadaan Awal": trans_names,
    "Peluang Sembuh (Fraksi)": [str(u_r_sol[k]) for k in u_r_keys],
    "Peluang Sembuh (%)": [f"{float(u_r_sol[k])*100:.2f}%" for k in u_r_keys],
    "Peluang Meninggal (Fraksi)": [str(u_d_sol[k]) for k in u_d_keys],
    "Peluang Meninggal (%)": [f"{float(u_d_sol[k])*100:.2f}%" for k in u_d_keys],
    "Total Probabilitas": [float(u_r_sol[rk]) + float(u_d_sol[dk]) for rk, dk in zip(u_r_keys, u_d_keys)]
})
display(df_u_summary)


Solusi Fraksi Eksak Peluang Sembuh:
{u_1^{A1}: 14663/15000, u_2^{A1}: 863/1000, u_3^{A1}: 2113/3750}
     Keadaan Awal  ... Total Probabilitas
0     Ringan (T1)  ...                1.0
1     Sedang (T2)  ...                1.0
2  Berat/ICU (T3)  ...                1.0

[3 rows x 6 columns]


## 5. Validasi Silang Menggunakan Teori Matriks Fundamental Kemeny-Snell

Dalam teori rantai Markov penyerap, kuantitas penyerapan dihitung serentak melalui:
1. Matriks Fundamental: $N = (I - Q)^{-1}$
2. Ekspektasi Waktu Penyerapan: $\\mathbf{v} = N \\mathbf{1}$
3. Matriks Probabilitas Penyerapan: $B = N R$


In [6]:
N_sym = model["N_sym"]
v_sym = model["v_sym"]
B_sym = model["B_sym"]

print("Matriks Fundamental N = (I - Q)^(-1):")
sp.pprint(N_sym)

print("\nVektor Waktu Rata-rata v = N * 1:")
sp.pprint(v_sym)

print("\nMatriks Probabilitas Penyerapan B = N * R:")
sp.pprint(B_sym)

num_res = fsa.solve_fsa_numerical(model["P_float"][:3, :3], model["P_float"][:3, 3:])
diff_v = np.max(np.abs(num_res["v"] - np.array([float(x) for x in v_sym])))
diff_B = np.max(np.abs(num_res["B"] - np.array([[float(val) for val in row] for row in B_sym.tolist()])))

print(f"\nSelisih Maksimum Numerik vs Simbolik pada v: {diff_v:.2e}")
print(f"Selisih Maksimum Numerik vs Simbolik pada B: {diff_B:.2e}")
print("Validasi Silang: 100% IDENTIK DAN TEPAT PRESISI!")


Matriks Fundamental N = (I - Q)^(-1):
[101             ]
[---   1/10  1/75]
[ 75             ]
[                ]
[1/5   3/2   1/5 ]
[                ]
[            104 ]
[4/75  2/5   --- ]
[             75 ]

Vektor Waktu Rata-rata v = N * 1:
[73]
[--]
[50]
[  ]
[19]
[--]
[10]
[  ]
[46]
[--]
[25]

Matriks Probabilitas Penyerapan B = N * R:
[14663   337 ]
[-----  -----]
[15000  15000]
[            ]
[863    137  ]
[----   ---- ]
[1000   1000 ]
[            ]
[2113   1637 ]
[----   ---- ]
[3750   3750 ]

Selisih Maksimum Numerik vs Simbolik pada v: 4.44e-16
Selisih Maksimum Numerik vs Simbolik pada B: 2.22e-16
Validasi Silang: 100% IDENTIK DAN TEPAT PRESISI!


## 6. Visualisasi Hasil Analisis Penyerapan


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

B_float = np.array([[float(val) for val in row] for row in B_sym.tolist()])
df_B_plot = pd.DataFrame(B_float, index=trans_names, columns=absorb_names)

sns.heatmap(df_B_plot, annot=True, fmt=".4f", cmap="Blues", cbar=True, ax=axes[0], vmin=0, vmax=1)
axes[0].set_title("Peluang Penyerapan Akhir (Matriks B)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Keadaan Awal (Transient)")
axes[0].set_xlabel("Keadaan Akhir (Absorbing)")

colors = ["#2ecc71", "#f39c12", "#e74c3c"]
bars = axes[1].bar(trans_names, v_days, color=colors, width=0.55, edgecolor="black", alpha=0.85)
axes[1].set_title("Ekspektasi Durasi Rawat sampai Diserap (Hari)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Rata-rata Waktu (Hari)")
axes[1].set_ylim(0, 18)

for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.3, f"{yval:.1f} hari\\n({yval/7:.2f} mgg)", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/hasil_fsa_covid.png", dpi=300)
plt.close(fig)
print("Grafik hasil FSA berhasil disimpan di figures/hasil_fsa_covid.png")


Grafik hasil FSA berhasil disimpan di figures/hasil_fsa_covid.png
